# Hacker Statistics: Random Walk & Randomization in Python
### Personal Study Notes — DataCamp Case Study

---

> **Core Question this chapter answers:**  
> *If I simulate 500 random walks, what is the probability of ending above step 60?*  
> Instead of solving this with complex math — we simulate it and count. That's **Hacker Statistics**.


## 📦 Imports

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

---
## Step 1: Randomness in Python

### 🧠 Feynman Explanation
Imagine a bag with 6 balls numbered 1 to 6. You close your eyes, grab one, note the number, put it back.  
That's random — you can't predict what you'll get.

Computers can't be **truly random** — they use a formula that *feels* random.  
If you tell that formula where to start (the **seed**), you get the same sequence every time.

> Think of seed like a shuffled playlist — same seed = same song order every time.

### `rand()` vs `randint()`

| Function | Returns | Use case |
|---|---|---|
| `np.random.rand()` | Decimal: `0.374...` | Probability checks (0 to 1) |
| `np.random.randint(1,7)` | Whole number: `1,2,3,4,5,6` | Simulating a dice 🎲 |

> Note: `randint(1, 7)` — the **7 is NOT inclusive**. Returns 1 to 6 only.

In [ ]:
# Seed fixes the starting point — same results every run
np.random.seed(123)
print("Random float:", np.random.rand())          # between 0 and 1
print("Dice roll:", np.random.randint(1, 7))      # 1 to 6
print("Dice roll:", np.random.randint(1, 7))      # different each call, same sequence

In [ ]:
# KEY INSIGHT: Resetting the seed rewinds the playlist
np.random.seed(5)
print(np.random.randint(1,7))  # X
print(np.random.randint(1,7))  # Y

np.random.seed(5)              # rewind!
print(np.random.randint(1,7))  # X again — same!
print(np.random.randint(1,7))  # Y again — same!

---
## Step 2: What is a Random Walk?

### 🧠 Feynman Explanation
Imagine you're standing in the middle of a long street at position **0**.  
You roll a dice every second:
- Roll **1 or 2** → step **backward**
- Roll **3, 4, or 5** → step **forward**
- Roll **6** → **jump** forward (lucky!)

After 100 rolls, where did you end up? That journey is a **random walk**.

### Real Life Examples

| Example | Step forward | Step back |
|---|---|---|
| 📈 Stock price | Good news → price up | Bad news → price down |
| 🦠 Virus spreading | Infects new person | Person recovers |
| 🏀 Basketball score | Team scores | Opponent scores |

> **Key insight:** Randomness doesn't mean "no reason exists."  
> It means **you can't know in advance.** From your perspective today, tomorrow's stock price is unknown = random.

### Why it drifts positive 📊
- 2/6 chance of going backward
- 4/6 chance of going forward (including the lucky jump)
- So over 100 steps → most likely end up **above 0**

In [ ]:
# One single step — the building block of everything
np.random.seed(123)
step = 50

dice = np.random.randint(1, 7)  # roll the dice

if dice <= 2:
    step = step - 1                        # go back
elif dice <= 5:
    step = step + 1                        # go forward
else:
    step = step + np.random.randint(1, 7)  # lucky jump — another random roll!

print(f"Dice: {dice}, New step: {step}")

---
## Step 3: Building the Full 100-Step Walk

### Two Key Concepts

**`random_walk` = Your Notebook 📓**  
Records every position you've ever been at: `[0, 1, 2, 1, 3, 4, ...]`

**`step` = Where You Are Right Now 🧍**  
Just one number — your current position.

### Why `random_walk[-1]`?
- `[-1]` = last item in the list = your current position
- You need to know where you ARE before you can move

### Why `max(0, step - 1)`?
- Prevents going below 0 (you can't be at a negative position in the park)
- `max(0, -1)` → returns `0` instead of `-1`

In [ ]:
np.random.seed(123)

random_walk = [0]       # 📓 notebook starts with just [0] — your starting position

for x in range(100):    # 🔄 take 100 steps
    
    step = random_walk[-1]              # 📍 where am I right now? (last page of notebook)
    dice = np.random.randint(1, 7)      # 🎲 roll the dice

    if dice <= 2:
        step = max(0, step - 1)         # go back (but never below 0)
    elif dice <= 5:
        step = step + 1                 # go forward
    else:
        step = step + np.random.randint(1, 7)  # lucky jump!

    random_walk.append(step)            # ✍️ write new position in notebook

print(f"Started at: 0")
print(f"Ended at: {random_walk[-1]}")
print(f"Total entries in notebook: {len(random_walk)}")  # 101: start + 100 steps

In [ ]:
# Visualize one walk
plt.figure(figsize=(10, 4))
plt.plot(random_walk)
plt.title("One Random Walk — 100 Steps")
plt.xlabel("Step number")
plt.ylabel("Position")
plt.show()

---
## Step 4: Simulating Multiple Walks

### 🧠 Feynman Explanation
Before: **one person** walking 100 steps.  
Now: **10 people** each walking 100 steps independently.

```
🗂️ all_walks   = filing cabinet (stores ALL journeys)
📓 random_walk = one person's notebook
```
After each person finishes → put their notebook in the cabinet.

### The Two Loops
```
for i in range(10):       ← OUTER loop: 10 different people
    random_walk = [0]     ← each person starts FRESH at 0
    for x in range(100):  ← INNER loop: each person takes 100 steps
        ...
    all_walks.append(random_walk)  ← file this person's journey
```

> ⚠️ `random_walk = [0]` must be **inside** the outer loop!  
> Otherwise every person continues from where the last one stopped.

In [ ]:
np.random.seed(123)

all_walks = []              # 🗂️ empty filing cabinet

for i in range(10):         # 10 people
    random_walk = [0]       # each starts fresh — MUST be inside outer loop!
    
    for x in range(100):    # 100 steps each
        step = random_walk[-1]
        dice = np.random.randint(1,7)
        if dice <= 2:
            step = max(0, step - 1)
        elif dice <= 5:
            step = step + 1
        else:
            step = step + np.random.randint(1,7)
        random_walk.append(step)
    
    all_walks.append(random_walk)   # 📁 file this person's notebook

print(f"Number of walks: {len(all_walks)}")           # 10
print(f"Steps per walk: {len(all_walks[0])}")         # 101 (start + 100 steps)

---
## Step 5: Visualizing All Walks — Why Transpose?

### The Shape Problem

```
all_walks shape:   (10, 101)  →  10 walks, each 101 steps
                   ↑ rows are walks — matplotlib plots columns!

After transpose:   (101, 10)  →  101 time points, each has 10 walkers
                   ↑ rows are time points — now each column = one walk ✅
```

> Think of it like rotating a table 90 degrees so matplotlib can read it correctly.

In [ ]:
np_aw = np.array(all_walks)
print(f"Shape before transpose: {np_aw.shape}")   # (10, 101)

# Without transpose — wrong plot (each row plotted separately)
plt.figure(figsize=(10, 4))
plt.plot(np_aw)
plt.title("Without Transpose — Looks Wrong")
plt.show()
plt.clf()

# With transpose — correct! Each column = one walker's journey over time
np_aw_t = np.transpose(np_aw)
print(f"Shape after transpose:  {np_aw_t.shape}")  # (101, 10)

plt.figure(figsize=(10, 4))
plt.plot(np_aw_t)
plt.title("With Transpose — Each Line = One Person's Walk ✅")
plt.xlabel("Step number")
plt.ylabel("Position")
plt.show()

---
## Step 6: Clumsiness — Random Falls

### 🧠 Feynman Explanation
Our walkers so far are **perfect** — they never fall.  
But in real life, sometimes randomly, for no reason, you **fall and go back to zero.**

That's clumsiness: a rare random event that resets everything.

### Implementation Logic
- Use `np.random.rand()` → gives a number between 0 and 1
- Set a **very small threshold** (0.001 = 0.1% chance)
- If random number falls below it → reset to 0

| Threshold | Probability | Falls per 100 steps |
|---|---|---|
| `0.1` | 10% | ~10 falls — too clumsy! |
| `0.005` | 0.5% | ~1 fall every 200 steps |
| `0.001` | 0.1% | ~1 fall every 1000 steps |

> `rand()` is used here, NOT `dice` — clumsiness is a **separate** random event from movement.

In [ ]:
np.random.seed(123)
all_walks = []

for i in range(250):
    random_walk = [0]
    for x in range(100):
        step = random_walk[-1]
        dice = np.random.randint(1,7)
        if dice <= 2:
            step = max(0, step - 1)
        elif dice <= 5:
            step = step + 1
        else:
            step = step + np.random.randint(1,7)

        # 🤕 Clumsiness: separate random event, very rare
        if np.random.rand() <= 0.001:   # 0.1% chance
            step = 0                     # fall! back to zero

        random_walk.append(step)
    all_walks.append(random_walk)

np_aw_t = np.transpose(np.array(all_walks))
plt.figure(figsize=(10, 5))
plt.plot(np_aw_t)
plt.title("250 Random Walks With Clumsiness")
plt.xlabel("Step number")
plt.ylabel("Position")
plt.show()

---
## Step 7: The Grand Finale — Distribution

### 🧠 The Big Question
> *"After 500 simulations, what's the probability of ending above step 60?"*

### Understanding `np_aw_t[-1, :]`

Think of `np_aw_t` as an Excel sheet:

```
         walk1  walk2  walk3  ...  walk500
step0  [   0,     0,     0,  ...,    0   ]
step1  [   1,     2,     0,  ...,    1   ]
...           
step100[ 45,    78,    23,  ...,   91   ]  ← LAST ROW
```

```python
np_aw_t[-1, :]
#       ↑    ↑
#   last row  all columns (all 500 walkers)
```

> 🏃 **Park analogy:** 500 people all walking simultaneously.  
> You blow a whistle — everyone freezes.  
> You write down everyone's position. That list is `ends`.

### What does the histogram tell you?
- X axis = final position after 100 steps
- Y axis = how many of the 500 walkers ended there
- From this you can calculate real probability!

In [ ]:
np.random.seed(123)
all_walks = []

for i in range(500):            # 500 simulations
    random_walk = [0]
    for x in range(100):
        step = random_walk[-1]
        dice = np.random.randint(1,7)
        if dice <= 2:
            step = max(0, step - 1)
        elif dice <= 5:
            step = step + 1
        else:
            step = step + np.random.randint(1,7)
        if np.random.rand() <= 0.001:
            step = 0
        random_walk.append(step)
    all_walks.append(random_walk)

np_aw_t = np.transpose(np.array(all_walks))

# 🎯 The key line — final position of ALL 500 walkers
ends = np_aw_t[-1, :]          # last row, all 500 columns

print(f"Shape of np_aw_t: {np_aw_t.shape}")    # (101, 500)
print(f"Number of final positions: {len(ends)}") # 500
print(f"Min position: {ends.min()}")
print(f"Max position: {ends.max()}")
print(f"Average final position: {ends.mean():.1f}")

In [ ]:
# Plot the distribution
plt.figure(figsize=(10, 5))
plt.hist(ends, bins=20, edgecolor='black')
plt.title("Distribution of Final Positions After 500 Walks")
plt.xlabel("Final position after 100 steps")
plt.ylabel("Number of walks")
plt.axvline(x=60, color='red', linestyle='--', label='Step 60')
plt.legend()
plt.show()

# Answer the real question!
prob = np.mean(ends >= 60)
print(f"\n🎯 Probability of ending at step 60 or above: {prob*100:.1f}%")
print(f"   ({np.sum(ends >= 60)} out of 500 walks)")

---
## 🗺️ Full Chapter Summary

| Concept | What it is | Key code |
|---|---|---|
| `np.random.seed()` | Fixes randomness for reproducibility | `np.random.seed(123)` |
| `randint(1,7)` | Dice roll, 1-6 only (7 not inclusive) | `np.random.randint(1,7)` |
| `rand()` | Random float 0 to 1 | `np.random.rand()` |
| `random_walk` | Notebook of all positions | `random_walk = [0]` |
| `[-1]` | Last item in list = current position | `random_walk[-1]` |
| `max(0, step-1)` | Floor at 0 — can't go negative | `max(0, step - 1)` |
| `all_walks` | Filing cabinet of all journeys | `all_walks.append(random_walk)` |
| Transpose | Rotate array so time goes down rows | `np.transpose(np_aw)` |
| Clumsiness | Rare reset event (0.1% chance) | `if rand() <= 0.001: step = 0` |
| `ends` | Final position of every walker | `np_aw_t[-1, :]` |
| `plt.hist(ends)` | Distribution of outcomes | Answers the probability question |

---

## 💡 Key Insight: Why This Matters for Data Science

> Instead of solving probability with complex math formulas,  
> **simulate it thousands of times and count the outcomes.**  
> This is called **Monte Carlo Simulation** — used in finance, ML, and science.

**Irreducible error:** Even the best ML model can never reach 100% accuracy  
because some randomness exists that no data can explain.  
In OLS this leftover randomness is called **residuals**.

---
*Study notes from DataCamp — Python Data Scientist track*